#### 0. LangChain에서 도구(tool) 활용 방법

In [2]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

gs
JF


In [3]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     base_url="https://api.groq.com/openai/v1",
#     model="meta-llama/llama-4-scout-17b-16e-instruct",
#     #model="moonshotai/kimi-k2-instruct-0905",
#     temperature=0
# )

from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

c:\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


solar-pro


- [tool decorator](https://python.langchain.com/docs/how_to/custom_tools/#tool-decorator)를 사용하면 쉽게 도구를 만들 수 있습니다

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """더하기 숫자 a와 b를 더합니다."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """곱하기 숫자 a와 b를 곱합니다."""
    return a * b

print(type(add))
print(type(multiply))

<class 'function'>
<class 'langchain_core.tools.structured.StructuredTool'>


In [5]:
# @tool 데코레이터로 선언된 함수는 직접 호출할 수 없다.
# 직접 호출하면 TypeError: 'StructuredTool' object is not callable
# add(10,20)

- LLM을 호출했을 때와 도구를 사용했을 때의 차이를 알아봅니다

In [6]:
query = '3 곱하기 5는?'
llm_result = llm.invoke(query)

print(llm_result.content)

3 곱하기 5는 **15**입니다.  

간단한 계산 과정:  
3 × 5 = 15  

도움이 필요하시면 언제든지 물어보세요! 😊


- 도구 리스트는 LLM에 해당하는 `BaseModel` 클래스에 `bind_tools` 메서드를 통해 전달합니다

In [7]:
llm_with_tools = llm.bind_tools([add, multiply])

print(type(llm_with_tools))
print(llm_with_tools)

<class 'langchain_core.runnables.base.RunnableBinding'>
bound=ChatUpstage(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002186539EC00>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000218653EECF0>, root_client=<openai.OpenAI object at 0x0000021835D39E20>, root_async_client=<openai.AsyncOpenAI object at 0x0000021864ED9910>, model_name='solar-pro', temperature=0.5, model_kwargs={}, disabled_params={'parallel_tool_calls': None}, upstage_api_key=SecretStr('**********'), upstage_api_base='https://api.upstage.ai/v1') kwargs={'tools': [{'type': 'function', 'function': {'name': 'add', 'description': '더하기 숫자 a와 b를 더합니다.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': '곱하기 숫자 a와 b를 곱합니다.', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type

- `AIMessage`의 `additional_kwargs` 속성은 `tool_calls`를 포함합니다
- `tool_calls`는 도구를 호출하는 메시지를 포함합니다

In [8]:
"""
AIMessage(
content='[질문 "3 곱하기 5는?"에 답하기 위해 곱셈 연산이 직접적으로 필요합니다. 
         제공된 함수 중 \'multiply\' 함수가 유일하게 필요한 계산을 수행할 수 있으므로 호출합니다.]', 
additional_kwargs={'refusal': None}, 
response_metadata={
    'token_usage': {'completion_tokens': 53, 'prompt_tokens': 578, 'total_tokens': 631, 
                    'completion_tokens_details': {'accepted_prediction_tokens': 0, 
                                                  'audio_tokens': 0, 'reasoning_tokens': 0, 
                                                  'rejected_prediction_tokens': 0}, 
                     'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 432}}, 
                     'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 
                     'system_fingerprint': None, 'id': '810b1e59-3322-4980-9ecc-16daf42fe59d', 
                     'finish_reason': 'tool_calls', 'logprobs': None}, 
                     id='lc_run--019be075-bde0-7323-885a-3075464b3746-0', 
                     tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 
                                  'id': 'chatcmpl-tool-63bddbf5f5ad4f179884c1a9258e5c7c', 'type': 'tool_call'}], 
                    invalid_tool_calls=[], 
                    usage_metadata={'input_tokens': 578, 'output_tokens': 53, 'total_tokens': 631, 
                                    'input_token_details': {'audio': 0, 'cache_read': 432}, 
                                    'output_token_details': {'audio': 0, 'reasoning': 0}}
)
"""

'\nAIMessage(\ncontent=\'[질문 "3 곱하기 5는?"에 답하기 위해 곱셈 연산이 직접적으로 필요합니다. \n         제공된 함수 중 \'multiply\' 함수가 유일하게 필요한 계산을 수행할 수 있으므로 호출합니다.]\', \nadditional_kwargs={\'refusal\': None}, \nresponse_metadata={\n    \'token_usage\': {\'completion_tokens\': 53, \'prompt_tokens\': 578, \'total_tokens\': 631, \n                    \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \n                                                  \'audio_tokens\': 0, \'reasoning_tokens\': 0, \n                                                  \'rejected_prediction_tokens\': 0}, \n                     \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 432}}, \n                     \'model_provider\': \'openai\', \'model_name\': \'solar-pro2-251215\', \n                     \'system_fingerprint\': None, \'id\': \'810b1e59-3322-4980-9ecc-16daf42fe59d\', \n                     \'finish_reason\': \'tool_calls\', \'logprobs\': None}, \n                     id=\'lc_run--019be075-bde0-

In [9]:
tool_result = llm_with_tools.invoke(query)

tool_result

AIMessage(content='[질문 "3 곱하기 5는?"에 직접 필요한 계산 결과를 제공하기 위해 `multiply` 함수를 호출합니다. 이 함수는 두 수의 곱셈을 수행하는 데 필수적입니다.]  \n\n정답: 15', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 582, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 560}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': 'c2b11c1b-e84c-4b8e-a199-baea00414750', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-e40d-7e43-b424-2cd9e2b0f1ea-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'chatcmpl-tool-a92ae654d1ec41be9de754c9563f7992', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 582, 'output_tokens': 61, 'total_tokens': 643, 'input_token_details': {'audio': 0, 'cache_read

In [10]:
from pprint import pprint

pprint(vars(tool_result))

{'additional_kwargs': {'refusal': None},
 'content': '[질문 "3 곱하기 5는?"에 직접 필요한 계산 결과를 제공하기 위해 `multiply` 함수를 호출합니다. 이 '
            '함수는 두 수의 곱셈을 수행하는 데 필수적입니다.]  \n'
            '\n'
            '정답: 15',
 'id': 'lc_run--019be0ea-e40d-7e43-b424-2cd9e2b0f1ea-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'tool_calls',
                       'id': 'c2b11c1b-e84c-4b8e-a199-baea00414750',
                       'logprobs': None,
                       'model_name': 'solar-pro2-251215',
                       'model_provider': 'openai',
                       'system_fingerprint': None,
                       'token_usage': {'completion_tokens': 61,
                                       'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                                                     'audio_tokens': 0,
                                                                     'reasoning_tokens': 0,
                          

In [11]:
tool_result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'chatcmpl-tool-a92ae654d1ec41be9de754c9563f7992',
  'type': 'tool_call'}]

In [12]:
#query = "안녕하세요? 오늘의 날씨는 어떤가요?"
query = "10 더하기 5는 얼마인가요?"
tool_result = llm_with_tools.invoke(query)

tool_result.tool_calls

[{'name': 'add',
  'args': {'a': 10, 'b': 5},
  'id': 'chatcmpl-tool-7f9c88a4c4e24279b1eeb967c0a3283d',
  'type': 'tool_call'}]

In [13]:
query = '안녕하세요? 오늘의 날씨는 어떤가요?'
tool_result = llm_with_tools.invoke(query)

tool_result.tool_calls

[]

In [14]:
from typing import Sequence

from langchain_core.messages import AnyMessage, HumanMessage

#query = "20과 10을 나누기 결과는?"
query = "20과 10을 곱한 결과는?"
human_message = HumanMessage(query)
message_list: Sequence[AnyMessage] = [human_message] 

print(message_list)


[HumanMessage(content='20과 10을 곱한 결과는?', additional_kwargs={}, response_metadata={})]


- `tool_calls` 속성은 도구를 호출하는 메시지를 포함합니다
- `tool_calls`를 가진 `AIMessage`의 형태를 기억하기

In [15]:
# llm_with_tools 는 LLM과 Tool이 연동된 RunnableBinding 객체
ai_message = llm_with_tools.invoke(message_list)
ai_message

AIMessage(content="[질문에서 20과 10을 곱한 결과를 직접 요청했기 때문에 'multiply' 함수 호출이 필수적입니다. 이 함수는 주어진 두 숫자를 곱하여 정확한 답을 제공합니다.]", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 586, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 576}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': '19f61403-0d32-43c3-acd2-157532672634', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-f0bc-7542-bdc6-69c1e63cc142-0', tool_calls=[{'name': 'multiply', 'args': {'a': 20, 'b': 10}, 'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 586, 'output_tokens': 57, 'total_tokens': 643, 'input_token_details': {'audio': 0, 'cache_read': 576},

In [16]:
ai_message.tool_calls

[{'name': 'multiply',
  'args': {'a': 20, 'b': 10},
  'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506',
  'type': 'tool_call'}]

In [17]:
from pprint import pprint

message_list.append(ai_message)

pprint(message_list)

[HumanMessage(content='20과 10을 곱한 결과는?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="[질문에서 20과 10을 곱한 결과를 직접 요청했기 때문에 'multiply' 함수 호출이 필수적입니다. 이 함수는 주어진 두 숫자를 곱하여 정확한 답을 제공합니다.]", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 586, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 576}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': '19f61403-0d32-43c3-acd2-157532672634', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-f0bc-7542-bdc6-69c1e63cc142-0', tool_calls=[{'name': 'multiply', 'args': {'a': 20, 'b': 10}, 'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 586, 'output_to

- `AIMessage`의 `tool_calls`를 활용해서 도구를 직접 호출할 수도 있습니다

In [18]:
ai_message.tool_calls

[{'name': 'multiply',
  'args': {'a': 20, 'b': 10},
  'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506',
  'type': 'tool_call'}]

In [19]:
tool_message = multiply.invoke(ai_message.tool_calls[0])

print(tool_message)

content='200' name='multiply' tool_call_id='chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506'


- 하지만 에이전트의 경우 도구를 직접 호출하는 것이 아니라 도구를 호출하는 메시지를 만들어서 전달합니다

In [20]:
message_list.append(tool_message)

pprint(message_list)

[HumanMessage(content='20과 10을 곱한 결과는?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="[질문에서 20과 10을 곱한 결과를 직접 요청했기 때문에 'multiply' 함수 호출이 필수적입니다. 이 함수는 주어진 두 숫자를 곱하여 정확한 답을 제공합니다.]", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 586, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 576}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': '19f61403-0d32-43c3-acd2-157532672634', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-f0bc-7542-bdc6-69c1e63cc142-0', tool_calls=[{'name': 'multiply', 'args': {'a': 20, 'b': 10}, 'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 586, 'output_to

In [21]:
tool_result = llm_with_tools.invoke(message_list)

pprint(tool_result)

AIMessage(content='20과 10을 곱한 결과는 **200**입니다.  \n\n계산 과정:  \n`20 × 10 = 200`  \n\n함수 호출 결과:  \n`\n\n` → `200`  \n\n따라서 정답은 **200**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 659, 'total_tokens': 738, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 560}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': '3cac46b9-3444-41f9-b14b-744993b9fea5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-f40c-73f2-8451-9980cb3a8456-0', tool_calls=[{'name': 'multiply', 'args': {'a': 20, 'b': 10}, 'id': 'chatcmpl-tool-21878e6280b941f486be34844f57b850', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 659, 'output_tokens': 79, 'total_tokens': 738, 'input_token_details': {'audio': 

- `message_list`의 순서를 기억하기

In [22]:
message_list

[HumanMessage(content='20과 10을 곱한 결과는?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="[질문에서 20과 10을 곱한 결과를 직접 요청했기 때문에 'multiply' 함수 호출이 필수적입니다. 이 함수는 주어진 두 숫자를 곱하여 정확한 답을 제공합니다.]", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 586, 'total_tokens': 643, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 576}}, 'model_provider': 'openai', 'model_name': 'solar-pro2-251215', 'system_fingerprint': None, 'id': '19f61403-0d32-43c3-acd2-157532672634', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019be0ea-f0bc-7542-bdc6-69c1e63cc142-0', tool_calls=[{'name': 'multiply', 'args': {'a': 20, 'b': 10}, 'id': 'chatcmpl-tool-a71e4cb9c55a40369ddb4c4b44a6d506', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 586, 'output_to